# 3. Evaluación

`evaluar.py` mide cualquier checkpoint con los **20 tickets de prueba**, escritos a mano y nunca vistos al entrenar,
con las métricas del benchmark de [Pondera](https://github.com/zamax14/Laya-Showcase):

- **Categoría** (`choice`), **prioridad** exacta y a ±1 nivel (`score`), **bloqueo** con umbral 0,5 y Brier (`noul`).
- **Semáforo:** aciertos de categoría según su confianza (verde > 80 %, amarillo 60–80 %, rojo < 60 %).
- **ECE:** distancia media entre la confianza de la categoría y su acierto real (0 es perfecto).

La validación mide si aprendió la tarea; los tickets de prueba, si eso se traslada a textos escritos por otra persona.
Si la validación sube y la prueba no, aprendió el estilo del generador, no la tarea.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import modelo  # Antes que torch: caché de Hugging Face en el proyecto y llaves locales.

import json
from IPython.display import Markdown, SVG, display

import evaluar

In [ ]:
# Cada versión es una carpeta de checkpoint; «base» es laya-multilingual sin ajustar.
!cd {ROOT} && {sys.executable} evaluar.py base v3=.model-cache/laya-mesa-de-ayuda-v3 v4=.model-cache/laya-mesa-de-ayuda

In [ ]:
results = json.loads(evaluar.RESULTS.read_text())
display(Markdown(evaluar.table({name: m["summary"] for name, m in results["modelos"].items()})))
display(SVG(filename=str(evaluar.CHART)))

## Qué cambió ticket por ticket

In [ ]:
names = list(results["modelos"])
first, last = results["modelos"][names[0]]["rows"], results["modelos"][names[-1]]["rows"]
for a, b in zip(first, last):
    if (a["category"], a["priority"]) != (b["category"], b["priority"]):
        ok = lambda r: "✓" if r["category"] == r["expected_category"] else "✗"
        print(f"{a['id']} {a['title'][:40]:40} ref {a['expected_category'] or 'ambiguo':10} | "
              f"{names[0]}: {a['category']} {ok(a)} {a['category_confidence']:.0f} % → {names[-1]}: {b['category']} {ok(b)} {b['category_confidence']:.0f} %")

## Probar un ticket propio

In [ ]:
from tarea import QUESTIONS

agent = modelo.load(ROOT / ".model-cache" / "laya-mesa-de-ayuda")
ticket = {"ticket": "Desde esta mañana nadie del almacén puede imprimir etiquetas de envío: la impresora Zebra "
                    "muestra «Error de cabezal» y ya la reiniciamos dos veces. Hoy salen 80 pedidos."}
answers = agent.predict(ticket, QUESTIONS)["answers"]
print("categoría:", answers["categoria"]["choice"], f"({100 * answers['categoria']['confidence']:.0f} %)")
print("prioridad:", round(answers["prioridad"]["score"], 2), "· bloqueo:", round(answers["bloqueo"]["noul"], 2))